# NB1 · Fundamentos del ecosistema Hugging Face y arquitecturas

Notebook **liviano** (corre en CPU o en una GPU T4). Recorre las herramientas del ecosistema y las tres familias de Transformer, **sin entrenar nada**. Es la primera clase práctica de la unidad.

**Contenido:** Pipeline API · AutoModel/AutoTokenizer · `datasets` · tokenización subpalabra · inferencia con encoder / decoder / encoder-decoder · Model Cards.

## 0. Preparación

En Colab: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU` (opcional para este NB).

In [1]:
%pip install -q -U transformers datasets huggingface_hub

In [2]:
import torch, transformers
print('transformers', transformers.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

transformers 5.17.0
device: cuda


In [3]:
# (Opcional) Montar Google Drive para guardar salidas. Usamos la carpeta 'Colab_Notebooks'.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab_Notebooks/UCA/IA_DL_II/Unidad_IV'
except Exception:
    BASE = './UCA/IA_DL_II/Unidad_IV'
import os; os.makedirs(BASE, exist_ok=True)
print('BASE =', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE = /content/drive/MyDrive/Colab_Notebooks/UCA/IA_DL_II/Unidad_IV


## 1. Pipeline API — inferencia en tres líneas

`pipeline("tarea")` arma en un objeto: preprocesamiento (tokenizer) → modelo → postprocesamiento.

In [4]:
from transformers import pipeline

clf = pipeline('sentiment-analysis')  # descarga un modelo por defecto (DistilBERT-SST2)
print(clf('Me encantó la clase de hoy'))
print(clf('El parcial estuvo durísimo y me fue mal'))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9972971081733704}]
[{'label': 'NEGATIVE', 'score': 0.8253583312034607}]


In [5]:
# Otras tareas con la misma API
ner = pipeline('token-classification', model='dslim/bert-base-NER', aggregation_strategy='simple')
print(ner('José dicta Aprendizaje Profundo en la Universidad de Palermo, en Buenos Aires.'))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER', 'score': np.float32(0.9927402), 'word': 'José', 'start': 0, 'end': 4}, {'entity_group': 'MISC', 'score': np.float32(0.62189615), 'word': '##endiza', 'start': 14, 'end': 20}, {'entity_group': 'ORG', 'score': np.float32(0.4015709), 'word': 'Prof', 'start': 23, 'end': 27}, {'entity_group': 'ORG', 'score': np.float32(0.97583723), 'word': 'la Universidad de Palermo', 'start': 35, 'end': 60}, {'entity_group': 'LOC', 'score': np.float32(0.9987968), 'word': 'Buenos Aires', 'start': 65, 'end': 77}]


In [6]:
# T5 es encoder-decoder (seq2seq). Lo usamos con AutoModelForSeq2SeqLM + generate,
# que es robusto entre versiones (algunas builds de transformers no registran el alias
# 'summarization'/'translation' en pipeline). Nota: t5-small entiende inglés.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained('t5-small')
t5_model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')

def t5_generate(prompt, max_new_tokens=40, num_beams=4):
    ids = t5_tok(prompt, return_tensors='pt').input_ids
    out = t5_model.generate(ids, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False)
    return t5_tok.decode(out[0], skip_special_tokens=True)

texto = ('Transfer learning pretrains a large model once on a huge corpus and then adapts it to many '
         'tasks with little data. It made NLP cheaper because you no longer train from scratch for '
         'every problem.')
print(t5_generate('summarize: ' + texto))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

transfer learning pretrains a large model once on a huge corpus and then adapts it to many tasks with little data. it made NLP cheaper because you no longer train from


## 2. ¿Qué hace la Pipeline por debajo?

Reproducimos la clasificación de sentimiento **a mano**: tokenizer → modelo → softmax. Es exactamente lo que la pipeline encapsula.

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

ckpt = 'distilbert-base-uncased-finetuned-sst-2-english'
tok = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(ckpt)

inputs = tok('I really loved this movie', return_tensors='pt')  # 1) tokenizar
with torch.no_grad():
    logits = model(**inputs).logits                             # 2) forward
probs = F.softmax(logits, dim=-1)                               # 3) postproceso
print('tokens:', tok.convert_ids_to_tokens(inputs['input_ids'][0]))
print('labels:', model.config.id2label)
print('probs :', probs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokens: ['[CLS]', 'i', 'really', 'loved', 'this', 'movie', '[SEP]']
labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
probs : tensor([[1.3351e-04, 9.9987e-01]])


## 3. AutoModel vs. AutoModelForXxx y el `config.json`

`AutoModel` da el **cuerpo** (backbone, sin cabeza); las variantes `ForXxx` agregan la **cabeza** de la tarea.

In [8]:
from transformers import AutoModel, AutoConfig

cfg = AutoConfig.from_pretrained('bert-base-uncased')
print('arquitectura:', cfg.model_type, '| capas:', cfg.num_hidden_layers, '| hidden:', cfg.hidden_size)

backbone = AutoModel.from_pretrained('bert-base-uncased')
out = backbone(**tok('un ejemplo cualquiera', return_tensors='pt'))
print('salida del backbone (last_hidden_state):', out.last_hidden_state.shape)  # (1, n_tokens, hidden)

arquitectura: bert | capas: 12 | hidden: 768


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


salida del backbone (last_hidden_state): torch.Size([1, 11, 768])


## 4. La librería `datasets`

Una línea para traer datos del Hub; `.map()` tokeniza en lotes con **caché** automático.

In [10]:
from datasets import load_dataset

ds = load_dataset('cornell-movie-review-data/rotten_tomatoes')   # reseñas de cine (pos/neg)
print(ds)
print(ds['train'][0])

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [11]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
def tok_fn(batch):
    return tokenizer(batch['text'], truncation=True)

ds_tok = ds.map(tok_fn, batched=True)   # la 2da vez, usa caché y no recalcula
print(ds_tok['train'].column_names)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


## 5. Tokenización subpalabra: BPE vs. WordPiece vs. SentencePiece

La misma frase, tres algoritmos. Observá: los `##` de WordPiece (BERT), el `Ġ`/espacio de BPE (GPT-2) y el `▁` de SentencePiece (T5).

In [12]:
frases = ['tokenización', 'antidisestablishmentarianism', 'aprendizaje profundo']
toks = {
    'BPE (gpt2)':        AutoTokenizer.from_pretrained('gpt2'),
    'WordPiece (bert)':  AutoTokenizer.from_pretrained('bert-base-uncased'),
    'SentencePiece (t5)':AutoTokenizer.from_pretrained('t5-small'),
}
for f in frases:
    print('\n=>', f)
    for nombre, t in toks.items():
        print(f'  {nombre:20s}', t.tokenize(f))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


=> tokenización
  BPE (gpt2)           ['token', 'iz', 'aci', 'Ã³n']
  WordPiece (bert)     ['token', '##iza', '##cion']
  SentencePiece (t5)   ['▁token', 'iza', 'ción']

=> antidisestablishmentarianism
  BPE (gpt2)           ['ant', 'idis', 'establishment', 'arian', 'ism']
  WordPiece (bert)     ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']
  SentencePiece (t5)   ['▁anti', 'd', 'is', 'est', 'abl', 'ish', 'ment', 'aria', 'nism']

=> aprendizaje profundo
  BPE (gpt2)           ['ap', 'rend', 'iz', 'aj', 'e', 'Ġprof', 'undo']
  WordPiece (bert)     ['apr', '##end', '##iza', '##je', 'prof', '##und', '##o']
  SentencePiece (t5)   ['▁', 'a', 'pre', 'nd', 'iza', 'je', '▁profund', 'o']


## 6. Las tres familias de Transformer en acción

**Encoder** (comprender) · **Decoder** (generar) · **Encoder-decoder** (transformar).

In [13]:
# Encoder (BERT): rellenar la máscara — tarea de comprensión
fill = pipeline('fill-mask', model='bert-base-uncased')
for r in fill('Paris is the [MASK] of France.')[:3]:
    print(round(r['score'],3), r['token_str'])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.997 capital
0.001 heart
0.0 center


In [14]:
# Decoder (GPT-2): generación autorregresiva
gen = pipeline('text-generation', model='gpt2')
print(gen('Deep learning is', max_new_tokens=30, do_sample=False)[0]['generated_text'])

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Deep learning is a very powerful tool for learning about the world around us. It's a tool that can help us understand the world around us, and it's a


In [15]:
# Encoder-decoder (T5): traducción (seq2seq), reutilizando t5_generate de la sección 1
print(t5_generate('translate English to German: The cat is sleeping on the sofa.'))

Die Katze schlaft auf dem Sofa.


## 7. Model Cards

La "ficha técnica" del modelo (README del repo): usos, datos, métricas, limitaciones y licencia.

In [16]:
from huggingface_hub import ModelCard
card = ModelCard.load('distilbert-base-uncased-finetuned-sst-2-english')
print(card.text[:800])   # primeras líneas de la Model Card

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]


# DistilBERT base uncased finetuned SST-2

## Table of Contents
- [Model Details](#model-details)
- [How to Get Started With the Model](#how-to-get-started-with-the-model)
- [Uses](#uses)
- [Risks, Limitations and Biases](#risks-limitations-and-biases)
- [Training](#training)

## Model Details
**Model Description:** This model is a fine-tune checkpoint of [DistilBERT-base-uncased](https://huggingface.co/distilbert-base-uncased), fine-tuned on SST-2.
This model reaches an accuracy of 91.3 on the dev set (for comparison, Bert bert-base-uncased version reaches an accuracy of 92.7).
- **Developed by:** Hugging Face
- **Model Type:** Text Classification
- **Language(s):** English
- **License:** Apache-2.0
- **Parent Model:** For more details about DistilBERT, we encourage users to check out [t
